In [1]:
import re
from pathlib import Path
from collections import Counter

BASE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\new\press\buildup_output\clips_buildup")
OUT = Path("prompts"); OUT.mkdir(exist_ok=True)
FOLDERS = {"lost_under_pressure": "LOST", "kept_under_pressure": "KEPT"}

MAX_SINGLE_TOKENS = 100_000   # lower this if your LLM has a smaller context
CHUNK_TOKENS = 40_000         # size of each part if splitting is needed

est = lambda s: len(s) // 4   # rough token estimate

# LOAD ALL FILES
records = []
for folder, label in FOLDERS.items():
    for f in sorted((BASE / folder).glob("*.txt")):
        m = re.match(r"(b\d+)_([A-Z]+)_(\d\d-\d\d)", f.stem)
        cid, team, t = m.groups() if m else (f.stem, "UNK", "")
        text = f.read_text(encoding="utf-8").strip()
        records.append(dict(id=cid, team=team, time=t, outcome=label,
                            block=f"### CLIP {cid} | team={team} | outcome={label} | time={t}\n{text}\n"))

counts = Counter((r["team"], r["outcome"]) for r in records)
counts_str = "\n".join(f"- {team} {out}: {n} clips" for (team, out), n in sorted(counts.items()))

RULES = """RULES:
- Use only information in the clips. Do not invent facts.
- Every claim must cite clip IDs as evidence.
- Give counts (e.g. "in 12 of 30 LOST clips") and use the clip counts above as denominators.
- Mark a pattern "tentative" if it appears in only 1-2 clips.
- Compare LOST vs KEPT: presser count, closing speed, free options, closest-defender distance, block compactness."""

FORMAT = """OUTPUT FORMAT:
1. BAR: build-up under pressure (strengths / weaknesses); pressing (strengths / weaknesses)
2. ATM: build-up under pressure (strengths / weaknesses); pressing (strengths / weaknesses)
3. What separates LOST from KEPT clips (top 3-5 factors)
4. Tactical takeaways (3-5 bullets)
5. Limitations (sample size, what the data cannot show)"""

CONTEXT = f"""You are a football tactical analyst. Below are automated analyses of build-up clips under pressure
from a Barcelona (BAR) vs Atletico Madrid (ATM) match. The team code in the clip label is assumed to be the
team building up. LOST = ball lost under pressure, KEPT = ball kept under pressure.
Each analysis has VERDICT, WHY, COUNTERFACTUAL.

CLIP COUNTS:
{counts_str}
"""

total = sum(est(r["block"]) for r in records)
print(f"{len(records)} clips, ~{total:,} tokens")
print(counts_str)

if total <= MAX_SINGLE_TOKENS:
    prompt = (f"{CONTEXT}\nTASK: Read ALL clips and write a general conclusion about strengths and weaknesses.\n\n"
              f"{RULES}\n\n{FORMAT}\n\nCLIPS:\n\n" + "\n".join(r["block"] for r in records))
    (OUT / "final_prompt.txt").write_text(prompt, encoding="utf-8")
    print("Wrote prompts/final_prompt.txt (single prompt)")
else:
    # split into parts
    parts, cur, cur_t = [], [], 0
    for r in records:
        t = est(r["block"])
        if cur and cur_t + t > CHUNK_TOKENS:
            parts.append(cur); cur, cur_t = [], 0
        cur.append(r); cur_t += t
    if cur: parts.append(cur)

    for i, p in enumerate(parts, 1):
        prompt = (f"{CONTEXT}\nTASK (part {i} of {len(parts)}): Extract recurring patterns ONLY from the clips below.\n"
                  f"For each pattern give: team, strength or weakness, build-up or pressing, a one-line description,\n"
                  f"the clip IDs that show it, and how many clips in this part show it. Do not draw final conclusions yet.\n\n"
                  f"{RULES}\n\nCLIPS:\n\n" + "\n".join(r["block"] for r in p))
        (OUT / f"map_prompt_{i:02d}.txt").write_text(prompt, encoding="utf-8")

    reduce = (f"{CONTEXT}\nBelow are pattern summaries from {len(parts)} parts covering all {len(records)} clips.\n"
              f"TASK: Merge them into one general conclusion about strengths and weaknesses. Add counts across parts "
              f"when the same pattern appears in several.\n\n{RULES}\n\n{FORMAT}\n\n"
              + "\n\n".join(f"--- PART {i} SUMMARY ---\n[PASTE OUTPUT OF map_prompt_{i:02d} HERE]" for i in range(1, len(parts)+1)))
    (OUT / "reduce_prompt.txt").write_text(reduce, encoding="utf-8")
    print(f"Too big for one prompt: wrote {len(parts)} map prompts + reduce_prompt.txt")

28 clips, ~15,410 tokens
- ATM KEPT: 3 clips
- ATM LOST: 11 clips
- BAR KEPT: 10 clips
- BAR LOST: 4 clips
Wrote prompts/final_prompt.txt (single prompt)


In [3]:
import re
from pathlib import Path

BASE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\new\press\buildup_output\clips_buildup")
OUT = Path("prompts"); OUT.mkdir(exist_ok=True)
FOLDERS = {"lost_under_pressure": "LOST", "kept_under_pressure": "KEPT"}

MAX_SINGLE_TOKENS = 100_000
CHUNK_TOKENS = 40_000
est = lambda s: len(s) // 4

# LOAD ALL FILES
records = []
for folder, label in FOLDERS.items():
    for f in sorted((BASE / folder).glob("*.txt")):
        m = re.match(r"(b\d+)_([A-Z]+)_(\d\d-\d\d)", f.stem)
        cid, team, t = m.groups() if m else (f.stem, "UNK", "")
        text = f.read_text(encoding="utf-8").strip()
        # clip ID and time are kept only so the file is traceable; the rules forbid citing them
        records.append(dict(block=f"### CLIP | team={team} | outcome={label}\n{text}\n"))

CONTEXT = """You are a football tactical analyst. Below are automated analyses of build-up clips under pressure
from a Barcelona (BAR) vs Atletico Madrid (ATM) match. The team code in each clip label is assumed to be the
team building up. LOST = the ball was lost under pressure, KEPT = the ball was kept under pressure.
Each analysis has VERDICT, WHY, COUNTERFACTUAL."""

RULES = """RULES:
- Write GENERAL conclusions only. Describe tactical tendencies and principles, not individual events.
- Do NOT give percentages, counts, ratios, success/failure rates, or win/loss statistics.
- Do NOT mention clip IDs, timestamps, player names, shirt numbers, or specific moments.
- Do NOT quote the clips or describe single situations. Speak at team level (e.g. "the press tends to...").
- Use only themes supported by the clips. Do not invent facts.
- Use qualitative wording such as "often", "tends to", "rarely" instead of numbers.
- You may contrast what generally separates LOST from KEPT situations (presser count, closing speed,
  free passing options, distance of the closest defender, block compactness), again without numbers."""

FORMAT = """OUTPUT FORMAT:
1. BAR: build-up under pressure (general strengths / weaknesses); pressing (general strengths / weaknesses)
2. ATM: build-up under pressure (general strengths / weaknesses); pressing (general strengths / weaknesses)
3. What generally separates a ball lost under pressure from a ball kept
4. General tactical takeaways (3-5 bullets)
5. Limitations (what this kind of data cannot show)"""

total = sum(est(r["block"]) for r in records)
print(f"{len(records)} clips, ~{total:,} tokens")

if total <= MAX_SINGLE_TOKENS:
    prompt = (f"{CONTEXT}\n\nTASK: Read ALL clips and write a general conclusion about the strengths "
              f"and weaknesses of each team.\n\n{RULES}\n\n{FORMAT}\n\nCLIPS:\n\n"
              + "\n".join(r["block"] for r in records))
    (OUT / "final_prompt.txt").write_text(prompt, encoding="utf-8")
    print("Wrote prompts/final_prompt.txt")
else:
    parts, cur, cur_t = [], [], 0
    for r in records:
        t = est(r["block"])
        if cur and cur_t + t > CHUNK_TOKENS:
            parts.append(cur); cur, cur_t = [], 0
        cur.append(r); cur_t += t
    if cur: parts.append(cur)

    for i, p in enumerate(parts, 1):
        prompt = (f"{CONTEXT}\n\nTASK (part {i} of {len(parts)}): Identify the general recurring tactical "
                  f"themes in the clips below, per team, as strengths or weaknesses, in build-up or pressing. "
                  f"Write short general statements only.\n\n{RULES}\n\nCLIPS:\n\n"
                  + "\n".join(r["block"] for r in p))
        (OUT / f"map_prompt_{i:02d}.txt").write_text(prompt, encoding="utf-8")

    reduce = (f"{CONTEXT}\n\nBelow are general theme summaries from {len(parts)} parts of the same match.\n"
              f"TASK: Merge them into one general conclusion about each team's strengths and weaknesses. "
              f"Themes that recur across parts should be emphasized in wording, not numbers.\n\n{RULES}\n\n{FORMAT}\n\n"
              + "\n\n".join(f"--- PART {i} SUMMARY ---\n[PASTE OUTPUT OF map_prompt_{i:02d} HERE]"
                            for i in range(1, len(parts) + 1)))
    (OUT / "reduce_prompt.txt").write_text(reduce, encoding="utf-8")
    print(f"Wrote {len(parts)} map prompts + reduce_prompt.txt")

28 clips, ~15,287 tokens
Wrote prompts/final_prompt.txt
